# Setup Dependedencies

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Install Pandas for formatting experiment results
import pandas as pd

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# Seed Test Data

In [7]:
texts_sentiment = [
    "I loved this product! Best purchase ever.",
    "This was terrible. I want a refund.",
    "The product was okay, nothing special.",
    "I loved the blue color of the sweater, however I noticed that the material had snagged.  I am sad, because I really loved it, but I cannot keep something so fragile.",
    "Je n'aime pas les chaussettes",
    "셔츠가 정말 예쁘다",  
    "Meh."
]

# human evaulation of the text_sentiments array (above)
human_expectations = {
    1: "Positive",
    2: "Negative",
    3: "Neutral",
    4: "Negative", 
    5: "Negative", 
    6: "Positive", 
    7: "Neutral"
}

# Prompt Experiments: /analyze-sentiment

In [ ]:
sentiment_variants = {
    "A": {
        "system": (
            "You are a precise sentiment analysis system. "
            "Given some input text, you must decide whether the overall sentiment is positive, negative, or neutral. "
            'Respond ONLY with a JSON object that has these keys: '
            '"label" (one of "positive", "negative", "neutral"), '
            '"confidence" (a number between 0 and 1), and '
            '"explanation" (a short explanation of why you chose this label).'
        )
    },
    "B": {
        "system": (
            "You are a precise sentiment analysis system. "
            "Treat texts that are mostly positive with minor complaints as positive. "
            "Treat texts that mix strong positives and strong negatives as neutral. "
            'Respond ONLY with JSON having keys "label", "confidence", "explanation" as described.'
        )
    },
    "C": {
        "system": (
            "You are a precise sentiment analysis system. "
            "In the explanation, briefly quote the phrases that indicate sentiment and explain how they lead to the label. "
            'Respond ONLY with JSON having keys "label", "confidence", "explanation".'
        )
    },
}

import json

def run_sentiment_experiments():
    results = []
    model = os.getenv("OPENAI_SENTIMENT_MODEL") or os.getenv("OPENAI_SUMMARIZE_MODEL", "gpt-4o-mini")
    for i, text in enumerate(texts_sentiment, start=1):
        for name, v in sentiment_variants.items():
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": v["system"]},
                    {"role": "user", "content": f"Analyze the sentiment of the following text:\n\n{text}"},
                ],
                response_format={"type": "json_object"},
            )
            data = json.loads((resp.choices[0].message.content or "{}").strip())
            results.append({"text_id": i, "variant": name, **data})
    return results

sentiment_results = run_sentiment_experiments()
sentiment_results  # inspect, compare to your own labels

# Format Results for Easy Comparison

In [ ]:

df_sentiment = pd.DataFrame(sentiment_results)

# Combine label + confidence into one string per row
df_sentiment["label_conf"] = df_sentiment.apply(
    lambda r: f"{r['label'].capitalize()} ({r['confidence']:.2f})",
    axis=1,
)
# Now pivot using the combined column
df_pivot = df_sentiment.pivot(index="text_id", columns="variant", values=["label", "confidence"])

formatted = pd.DataFrame(index=df_pivot.index)

for v in df_pivot.columns.levels[1]:  # variant names
    labels = df_pivot["label"][v]
    confs = df_pivot["confidence"][v]
    formatted[v] = labels.str.capitalize() + " (" + confs.round(2).astype(str) + ")"

formatted



formatted["Human"] = formatted.index.map(
    lambda tid: human_expectations.get(tid, "N/A")

)

formatted = formatted[["A", "B", "C", "Human"]]
formatted


,A,B,C,Human
text_id,,,,
1,Positive (0.95),Positive (0.95),Positive (0.95),Positive
2,Negative (0.95),Negative (0.95),Negative (0.95),Negative
3,Neutral (0.85),Neutral (0.85),Neutral (0.85),Neutral
4,Negative (0.85),Neutral (0.85),Mixed (0.85),Negative
5,Negative (0.9),Negative (0.95),Negative (0.95),Negative
6,Positive (0.95),Positive (0.95),Positive (0.95),Positive
7,Neutral (0.8),Neutral (0.85),Neutral (0.85),Neutral
